# kodoom: resumable Gemma 3 27B translation on Kaggle TPU

Active stage: **bulk-translation**, project-plan **v21**, section 1.2.
Translate all **1,200 train + 400 test typed cases (8,000 decisions)** and
a frozen **2,000-record helmo selection** with **gemma3-27b-tpu-bf16**.
End at **drafts-complete-awaiting-review**; every output remains unreviewed.

Use a **private** notebook, Internet enabled and **TPU v5e-8**. Accept Gemma's
terms on Kaggle and attach `google/gemma-3/flax/gemma3-27b-it/1` under Models.
Python 3.11–3.13 is supported. Create an existing **private HF dataset repo**
(use an owner-administered organization for collaborators) and enable the
`HF_TOKEN` Kaggle secret with write access. A private GitHub checkout also
needs `GITHUB_TOKEN`. Never paste credentials into cells.

Use **Save Version → Save & Run All** for a server-side run. Keep the same
campaign ID, code pin and storage repository in later sessions. Each operator
uses their own accounts, model access and tokens. See `docs/execution-workflow.md`.


In [ ]:
PROVIDER = "kaggle"
PROFILE = "kaggle-tpu"
BACKEND = "jax"
CODE_REVISION = "4d6af170cd05e64e31d3ffee0a433702c21fb043"
CHECKOUT = "/tmp/kodoom-tpu-code"
CAMPAIGN_ID = "gemma27b-bulk-v1"
HF_DATASET_REPO = ""  # REQUIRED: your private organization/dataset repo
OPERATOR = "owner"  # change this name for an authorized collaborator
TAKEOVER = False  # True only after confirming an interrupted prior worker ended

## Bootstrap the pinned runtime

Setup, model loading, compilation and generation share an eight-hour budget
starting here. The last 30 minutes are reserved for finalization. JAX probes
and inference run in child processes so the notebook kernel never owns TPUs.


In [ ]:
import contextlib
import os
import time
import types
import urllib.parse
import urllib.request

SESSION_STARTED = time.monotonic()
if not HF_DATASET_REPO:
    raise ValueError("Set HF_DATASET_REPO before starting the campaign")

if PROVIDER == "kaggle":
    from kaggle_secrets import UserSecretsClient

    with contextlib.suppress(Exception):
        os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
elif PROVIDER == "colab":
    from google.colab import userdata

    with contextlib.suppress(Exception):
        os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
url = (
    "https://api.github.com/repos/Farahani1/kodum/contents/src/kodoom/runtime.py"
    + "?ref="
    + urllib.parse.quote(CODE_REVISION, safe="")
)
headers = {"Accept": "application/vnd.github.raw+json"}
if os.environ.get("GITHUB_TOKEN"):
    headers["Authorization"] = "Bearer " + os.environ["GITHUB_TOKEN"]
with urllib.request.urlopen(urllib.request.Request(url, headers=headers)) as response:
    source = response.read().decode("utf-8")
runtime = types.ModuleType("kodoom_bootstrap")
exec(compile(source, "runtime.py", "exec"), runtime.__dict__)
checkout = runtime.bootstrap(PROVIDER, CODE_REVISION, CHECKOUT, backend=BACKEND)

## Restore, benchmark and translate pending work

The package verifies source scope and private storage with a save/restore
probe before loading weights. The first session benchmarks BF16 batches 1/2/4/8
with a memory reserve and freezes the fastest passing configuration. Resumes
reuse it. One worker loads weights once, buckets requests by length and saves
each complete case atomically. Helmo text is translated together as strict JSON.

Snapshots are uploaded and read back approximately every ten minutes, and at
normal stop. Thirty minutes of unsaved completed work stops new inference.
Hard termination can lose work since the last verified checkpoint.


In [ ]:
from kodoom.bulk.launch import run

result = run(
    provider=PROVIDER,
    profile=PROFILE,
    campaign_id=CAMPAIGN_ID,
    repo=HF_DATASET_REPO,
    operator=OPERATOR,
    takeover=TAKEOVER,
    session_started=SESSION_STARTED,
)

## Progress and review

Remote `campaigns/<campaign-id>/checkpoint.json` identifies a verified archive
with English inputs, complete-case shards, progress, events, production settings,
TPU metrics and review CSVs. Local artifacts are under
`/kaggle/working/kodoom/data/bulk/<campaign-id>/`. Save private notebook outputs
as an additional copy. Keep model weights and caches out of saved output.

Review `review/cases-*.csv` and `review/labels-*.csv` with the English/Persian
context. Empty review fields remain unreviewed; automatic findings are flags.
All 400 test cases need human review. A diagnostic sample cannot establish
dataset-wide label error prevalence or detector coverage. Adoption, training
and publication still require the research decisions in the project plan.

Read-only snapshot download and pause instructions are in `docs/execution-workflow.md`.
Actual TPU fit, speed and a second live session remain unverified until run.


In [ ]:
print("Status:", result["status"])
print("Completed / failed / total cases:", result["counts"])
print("Private artifacts:", result["artifact_root"])
print("Last verified remote revision:", result["last_verified_revision"])
print("Final remote save verified:", result["persisted"])
print("Stop here for human review; resume the same campaign if work remains.")